<a href="https://colab.research.google.com/github/syifaST/simuteh--2330511062-/blob/main/notebook/B5_praproses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys
REPO = "https://github.com/syifaST/simuteh--2330511062-.git"

if not os.path.exists("/content/simuteh"):
    !git clone {REPO} /content/simuteh
os.chdir("/content/simuteh")

for f in ["notebook", "hasil", "laporan"]:
    os.makedirs(f, exist_ok=True)

# buat ulang dataset kalau ada folder atau label.csv yang kurang
perlu = ["citra", "bersih", "mask_daun", "mask_bercak"]
if (not all(os.path.exists(f"data/sintetis/{d}") for d in perlu)
        or not os.path.exists("data/sintetis/label.csv")):
    !python src/buat_dataset_sintetis.py

os.chdir("/content/simuteh/notebook")
sys.path.append("../src")

print("Folder kerja:", os.getcwd())
print("Jumlah citra:", len(os.listdir("../data/sintetis/citra")))
print("Jumlah bersih:", len(os.listdir("../data/sintetis/bersih")))
print("Isi src:", os.listdir("../src"))

Cloning into '/content/simuteh'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (376/376), done.
remote: Compressing objects: 100% (358/358), done.
remote: Total 376 (delta 75), reused 4 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (376/376), 56.33 MiB | 18.26 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Selesai: 60 citra tersimpan di data/sintetis
Folder kerja: /content/simuteh/notebook
Jumlah citra: 61
Jumlah bersih: 61
Isi src: ['utils_cv.py', 'praproses.py', 'buat_dataset_sintetis.py']


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from utils_cv import baca, statistik, tampil, kontras_bercak
from praproses import praproses, simpan_param

# Parameter praproses
param = {
    "gamma": 1.0,
    "clahe_clip": 2.0,
    "penapis": "median",
    "kernel": 3
}

simpan_param(param, "../hasil/param_praproses.json")

# Memilih kanal terbaik berdasarkan Bagian 2
KANAL = 2  # Indeks BGR: 0 = B, 1 = G, 2 = R

# Membaca dataset
label = pd.read_csv("../data/sintetis/label.csv")
baris = []

for _, d in label[label.kelas != "sehat"].iterrows():
    img = baca(f"../data/sintetis/citra/{d.nama_file}")
    md = cv2.imread(f"../data/sintetis/mask_daun/{d.nama_file}", 0)
    mb = cv2.imread(f"../data/sintetis/mask_bercak/{d.nama_file}", 0)

    # Melakukan praproses
    out = praproses(img, param)

    # Menghitung kontras bercak sebelum dan sesudah praproses
    baris.append([
        d.kondisi_cahaya,
        kontras_bercak(img[:, :, KANAL], md, mb),
        kontras_bercak(out[:, :, KANAL], md, mb)
    ])

# Membuat tabel evaluasi
evaluasi = pd.DataFrame(
    baris,
    columns=["kondisi", "KB_sebelum", "KB_sesudah"]
)

print(evaluasi.groupby("kondisi").mean().round(2))

                KB_sebelum  KB_sesudah
kondisi                               
bayangan              5.09        8.64
gelap                10.40       17.48
kontras_rendah        3.91        4.99
normal                9.02       11.06
terang                6.64        5.84
